# 오차 전파 실습

**Error Propagation · 불확실성 전파**

입력의 불확실성이 계산 과정을 거쳐 출력 불확실성으로 전달되는 것을 추정하는 방법.

소재 분야에서 이해하기: 측정 두께 오차가 계산된 전도도 오차로 얼마나 커지는지 본다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [SciPy 준몬테카를로 문서](https://docs.scipy.org/doc/scipy/reference/stats.qmc.html)

## 1. 선형 근사와 몬테카를로 비교

작은 오차에서는 1차 근사가 잘 맞지만, 오차가 커지면 어긋납니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def measured_property(thickness, conductivity, length=1e-3, width=1e-4):
    return length / (conductivity * thickness * width)

t0, c0 = 500e-9, 1.2e6
base = measured_property(t0, c0)

def linear_propagation(sigma_t, sigma_c, step=1e-12):
    dt = (measured_property(t0 + step, c0) - measured_property(t0 - step, c0)) / (2 * step)
    dc = (measured_property(t0, c0 + 1.0) - measured_property(t0, c0 - 1.0)) / 2.0
    return np.sqrt((dt * sigma_t) ** 2 + (dc * sigma_c) ** 2)

print('기준 저항 %.3f Ohm' % base)

In [ ]:
for relative in (0.01, 0.05, 0.15, 0.30):
    sigma_t, sigma_c = relative * t0, relative * c0
    linear = linear_propagation(sigma_t, sigma_c)
    samples = measured_property(rng.normal(t0, sigma_t, 200000), rng.normal(c0, sigma_c, 200000))
    samples = samples[np.isfinite(samples) & (samples > 0)]
    print('상대오차 %4.0f%% -> 선형 근사 %.3f / 몬테카를로 %.3f / 몬테카를로 평균 %.3f'
          % (100 * relative, linear, samples.std(), samples.mean()))

## 2. 해석

저항은 두께와 전도도의 역수에 비례하므로 관계가 비선형입니다. 오차가 커지면 분포가 한쪽으로
치우쳐 평균 자체가 기준값에서 벗어나고, 1차 근사는 표준편차를 과소평가합니다.

In [ ]:
sigma = 0.30
samples = measured_property(rng.normal(t0, sigma * t0, 200000), rng.normal(c0, sigma * c0, 200000))
samples = samples[np.isfinite(samples) & (samples > 0) & (samples < 20 * base)]
plt.hist(samples, bins=100, density=True)
plt.axvline(base, color='k', ls='--', label='value at the nominal inputs')
plt.axvline(samples.mean(), color='r', label='mean of the propagated distribution')
plt.xlabel('resistance (Ohm)'); plt.legend(); plt.show()

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#error-propagation)을 여세요.